# 02 · Baseline and review

**Question:** Does adding rule/example context improve the lexical reference?

This notebook renders recorded **competition-data** evidence. It verifies the committed artifact hashes, not private out-of-fold predictions. No credentials, raw comments, weight downloads, or training are needed. Full private metric recomputation remains `uv run jigsaw review`.

In [1]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from jigsaw_rules.review import public_evidence
from jigsaw_rules.runtime import environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
baseline = public_evidence(root, "baseline")
semantic = public_evidence(root, "semantic")
assert baseline["training_sha256"] == semantic["training_sha256"]
print("Competition data | 2,029 training rows | recorded local cross-validation")
print("Verification: aggregate file checksums and provenance; no model fitting.")

names = {"comment_only": "Comment-only TF-IDF", "rule_examples": "Rule/example TF-IDF",
         "semantic_margin": "Frozen semantic margin", "semantic_classifier": "Semantic classifier"}
protocols = {"seen_rule": "Familiar rules", "heldout_rule": "Held-out rule"}

def metric_table(records):
    return pd.DataFrame([{"Model": names[r["model"]], "Validation": protocols[r["protocol"]],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"],
        "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"],
        "Average precision": r["metrics"]["average_precision"]} for r in records]).round(4)


Competition data | 2,029 training rows | recorded local cross-validation
Verification: aggregate file checksums and provenance; no model fitting.


## Controlled feature comparison
`comment_only` uses comment TF-IDF with logistic regression. `rule_examples` adds comment-to-rule/example cosine similarities, positive/negative maximum similarity, and their margin. Both are evaluated under the recorded split design. This is a lexical reference, not a claim of deep rule understanding.

In [2]:
display(metric_table(baseline["results"]))

,Model,Validation,Rule macro AUC,Log loss,Brier,Average precision
0,Comment-only TF-IDF,Familiar rules,0.7281,0.6148,0.2131,0.7170
1,Rule/example TF-IDF,Familiar rules,0.7287,0.6162,0.2139,0.7202
2,Comment-only TF-IDF,Held-out rule,0.6041,0.6731,0.2400,0.6108
3,Rule/example TF-IDF,Held-out rule,0.6156,0.6736,0.2405,0.6241


![Lexical validation comparison](../reports/baseline/comparison.svg)

## Held-out-rule feature effect
The difference below is descriptive. It is not a significance claim or a score from Kaggle.

In [3]:
held = {r["model"]: r["metrics"] for r in baseline["results"] if r["protocol"] == "heldout_rule"}
display(pd.DataFrame([{"Metric": metric, "Rule/example minus comment-only": held["rule_examples"][metric] - held["comment_only"][metric]}
    for metric in ["rule_macro_auc", "log_loss", "brier"]]).round(4))

,Metric,Rule/example minus comment-only
0,rule_macro_auc,0.0115
1,log_loss,0.0004
2,brier,0.0005


## Interpretation and reproducibility
Rule/example features modestly improve held-out ranking, while probability losses do not improve. The lexical model remains the reference for later candidates. Coefficients describe association, not causal effects. Raw examples and row-level errors are intentionally absent from the public notebook.

Training is explicit: `uv run jigsaw baseline --cloud`. That command fits or resumes a source-fingerprinted experiment; it is not required to read these results. Valid completed folds are reused, but an interrupted CPU solver restarts its active fold. Source changes can create a new experiment identity.

[03 · Results](03_saved_results.ipynb) is the consolidated employer overview.

## Next experiment · task-informed feature ablations

**Hypothesis:** a rule violation depends on a comment's relationship to the supplied policy and its positive/negative examples, not toxicity alone. Word overlap, character similarity, and contrastive support statistics probe different relationships. Structural cues can also be shortcuts, so they are tested separately.

Four predeclared candidates share the same training-only word vocabulary and saved, purged reference splits. They add (1) rule text similarity, (2) order-invariant positive/negative support contrasts, (3) writing structure, or (4) all groups. The combined candidate has 26 dense features plus the sparse comment representation. No validation labels enter vocabulary, scaling, or feature construction.

Run `.venv/bin/python -m jigsaw_rules.cli features --cloud --export` in your existing SageMaker checkout. This is a new CPU experiment, not a rerun of the lexical/Qwen baselines. It makes **no submission CSV**. Completed folds are resumed; S3 checkpoints follow each fold. The fixed reference remains unchanged.

In [4]:
import subprocess
import sys
from jigsaw_rules.features import feature_evidence

# Opt in only in your own configured workspace. Publication keeps this False.
RUN_FEATURE_EXPERIMENT = False
CHECKPOINT_TO_S3 = True
if RUN_FEATURE_EXPERIMENT:
    command = [sys.executable, "-m", "jigsaw_rules.cli", "features", "--export"]
    if CHECKPOINT_TO_S3:
        command.append("--cloud")
    subprocess.run(command, cwd=root, check=True)
feature_run = feature_evidence(root)
if feature_run is None:
    print("Feature ablation code is ready; no competition-data ablation results are published yet.")
else:
    feature_table = pd.DataFrame([{"Candidate": r["model"], "Protocol": r["protocol"],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"], "Log loss": r["metrics"]["log_loss"],
        "Brier": r["metrics"]["brier"], "Original fit seconds": r["fit_seconds"]}
        for r in feature_run["results"]])
    display(feature_table.round(4))
    display(pd.DataFrame(feature_run["audit"]).round(4))
    intervals = pd.DataFrame(feature_run["uncertainty"])
    display(intervals.loc[:, ["model", "protocol", "observed_delta", "ci_lower", "ci_upper"]].round(4))
    import matplotlib.pyplot as plt
    held = intervals.loc[intervals.protocol == "heldout_rule"].sort_values("observed_delta")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    positions = range(len(held))
    ax.hlines(positions, held.ci_lower, held.ci_upper)
    ax.scatter(held.observed_delta, positions)
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_yticks(list(positions), held.model.str.replace("_", " "))
    ax.set_xlabel("Held-out rule macro AUC difference from the fixed lexical reference")
    ax.set_title("Task-informed feature ablations | paired 95% group-bootstrap intervals")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    from io import StringIO
    from IPython.display import SVG
    buffer = StringIO()
    fig.savefig(buffer, format="svg")
    display(SVG(buffer.getvalue()))
    plt.close(fig)
    coefficients = pd.DataFrame(feature_run["coefficients"])
    combined = coefficients.loc[coefficients.model == "combined"]
    display(combined.groupby(["protocol", "feature"]).coefficient.agg(["mean", "min", "max"]).round(4))


Feature ablation code is ready; no competition-data ablation results are published yet.


### How to read the next results
Compare each candidate against the preserved rule/example reference on the **same rows**. Inspect held-out rule macro AUC first, then each rule, log loss, Brier score, and paired comment-group intervals. These four-candidate exploratory intervals are not multiplicity-adjusted; they do not certify a winner. No automatic promotion, calibration claim, or leaderboard claim is made.

Writing-structure associations and standardized dense coefficients are descriptive, not causal, and only two rules are observed. Coefficient ranges show sensitivity across folds; different regularization/scaling means this comparison is not solely a feature-count experiment. Candidate improvement here should motivate a separately validated joint rule/comment encoder, not a claim that lexical features solve arbitrary policy understanding.